# calculate duration of each activity

In [1]:
import os

import matplotlib.pyplot as plt

import numpy as np

from deepview.calculate_results.data.umineko_data import (
    read_umineko_data,
read_umineko_path,
extract_data_from_year_back,
label_dict,
get_accel_batch_data,
)

from collections import Counter

In [2]:
# label_dict = {
#     'ground_stationary': 0,
#     'stationary': 0,
#     'preening': 0,
#     'bathing': 1,
#     'bathing_poss': 1,
#     'flight_take_off': 2,
#     'flight_cruising': 3,
#     'flying_active': 4,
#     'flying_passive': 4,
#     'foraging': 5,
#     'poss_foraging': 5,
#     'foraging_fish_poss': 6,
#     'foraging_insect_poss': 7,
#     'forgaing_insect': 7,
#     'foraging_non-fish': 8,
#     'foraging_steal': 9,
#     'foraging_poss': 9,
#     'foraging_dive': 10,
#     'surface_seizing': 11,
#     'body_shaking': 12,
#     'ground_active': 13,
#     'unknown': 14,  #-1
# }

In [3]:
back_label_path = r'D:\logbot-data\BioTaggerData\masterLabelsByOtsuka\animal_id.csv'
umi_root_path = r'D:\logbot-data\BioTaggerData\export\raw\umineko\v.1.0.0'
file_paths = read_umineko_path(umi_root_path)

year = '2018'
data_path = r'D:\code\DeepView\deepview\calculate_results\data\umineko_%s.npy'%year
if os.path.exists(data_path):
    tmp = np.load(data_path, allow_pickle='TRUE').item()
    df_list = tmp['raw_data']
    df_clean_list = tmp['labeled_data']
    
selected_df, selected_clean_df = (
        extract_data_from_year_back(df_list, df_clean_list, 1))

selected_df['label_id'] = selected_df['label'].map(label_dict)
selected_clean_df['label_id'] = selected_clean_df['label'].map(label_dict)



C:\Users\dell\AppData\Local\Temp\ipykernel_6072\3663547461.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['label_id'] = selected_df['label'].map(label_dict)
C:\Users\dell\AppData\Local\Temp\ipykernel_6072\3663547461.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_clean_df['label_id'] = selected_clean_df['label'].map(label_dict)


In [4]:


# Define a function to identify segments based on value change
def segment_values(series):
    # Forward fill NaN values to preserve segments
    filled_series = series.ffill()
    
    # Identify changes in the value
    changes = filled_series.diff().ne(0).astype(int)
    
    # Assign a unique segment ID to each group of continuous values
    segments = changes.cumsum()
    
    # Mask the NaNs again to keep them separate
    segments[series.isna()] = np.nan
    
    return segments


## umineko 2018, back 1, whole data

In [5]:
df = selected_df[['label', 'label_id']].copy()
# Apply the function to create a new column with segment IDs
df['segment'] = segment_values(df['label_id'])

# Group segments by unique values
grouped_segments = df.dropna().groupby('label_id')['segment'].unique()

# Convert the grouped segments to a list of lists for better visualization
grouped_segments_list = {label: list(segments) for label, segments in grouped_segments.items()}

print("Grouped Segments:")
for label, segments in grouped_segments_list.items():
    print(f"Value {label}: Segments {segments}")

Grouped Segments:
Value 0.0: Segments [20940.0, 20942.0, 20944.0, 20946.0, 20954.0, 20956.0, 20958.0, 20964.0, 20966.0, 20968.0, 20971.0, 20973.0, 20979.0, 20981.0, 20983.0, 20991.0, 20993.0, 20995.0, 20998.0, 21000.0, 21004.0, 21006.0, 21013.0, 21015.0, 21018.0, 21020.0, 21026.0, 21028.0, 21030.0, 21032.0, 21034.0, 21036.0, 21038.0, 21040.0, 21043.0, 21046.0, 21055.0, 21058.0, 21071.0, 21073.0, 21077.0, 21079.0, 21083.0, 21085.0, 21088.0, 21090.0, 21092.0, 21095.0, 21116.0, 21121.0, 21124.0, 21132.0, 21134.0, 21136.0, 21138.0, 21142.0, 21144.0, 21146.0, 21148.0, 21160.0, 21162.0, 21164.0, 21166.0, 21175.0, 21206.0, 21208.0, 21210.0, 21220.0, 21222.0, 21237.0, 21247.0, 21249.0, 21251.0, 21256.0, 21258.0, 21260.0, 21262.0, 21264.0, 21266.0, 21268.0, 21272.0, 21274.0, 21276.0, 21278.0, 21281.0, 21283.0, 21285.0, 21293.0, 21295.0, 21297.0, 21299.0, 21301.0, 21303.0, 21305.0, 21307.0, 21309.0, 21314.0, 21321.0, 21331.0, 21335.0, 21338.0, 21345.0, 21347.0, 21349.0, 21351.0, 21353.0, 21355.0

## get segments of every activity

First, concatenate the segment ID (df) with raw df data.
1. when the segment is longer than the threshold (segment_len), split the segment without overlap, then padding the last part the same as segment_len.
2. when the segment is shorter than the segment_len, just padding the segment to the segment_len.

In [9]:
# selected_df['segment'] = df['segment'].values
selected_df.loc[:, 'segment'] = df['segment']

In [61]:
segment_len = 25 * 60  # 1 minute
select_columns = ['acc_x', 'acc_y', 'acc_z', 'label_id']
activity_duration_dict = {}
len_sw = step = segment_len
all_seg_data = []
for label, segments in grouped_segments_list.items():
    # print(f"Value {label}: Segments {segments}")
    for seg in segments:
        segdf = selected_df[selected_df['segment'] == seg].copy()
        segnp = segdf[select_columns].values  # segnp.shape = length, dim
        # get batch of data and pad data altomatically
        if segnp.shape[0] > segment_len:
            # generate batch of data by overlapping the training set
            data_batch = []
            for idx in range(0, segnp.shape[0], step):  # step10
                data_batch.append(segnp[idx: idx + len_sw, :])
            # data_batch.append(segnp[-1 - len_sw: -1, :])  # last batch
            if(data_batch[0].shape == data_batch[-1].shape):
                xlist = np.stack(data_batch, axis=0)  # [B, Len90, dim6]
            else:
                xlist = np.stack(data_batch[:-1], axis=0)
            # Create an array of ones as mask
            ones_array = np.ones((xlist.shape[0], segment_len, 1))
            # Concatenate the original array with the ones array along the last axis
            new_array = np.concatenate((xlist, ones_array), axis=2)
            
            if(data_batch[0].shape != data_batch[-1].shape):
                # padding
                zeros_array = np.zeros((1, segment_len, len(select_columns)+1))
                zeros_array[0,:data_batch[-1].shape[0],:-1] = data_batch[-1]  # raw data
                zeros_array[0,:,-2] = data_batch[-1][-1, -1]  # replace label column with original column
                zeros_array[0,:data_batch[-1].shape[0],-1] = 1  # replace mask column with 1
                
                new_array = np.concatenate((new_array, zeros_array), axis=0)
            
        else:
            new_array = np.zeros((1, segment_len, len(select_columns)+1))
            new_array[0,:segnp.shape[0],:-1] = segnp  # raw data
            new_array[0,:,-2] = segnp[-1, -1]  # replace label column with original column
            new_array[0,:segnp.shape[0],-1] = 1  # replace mask column with 1
        all_seg_data.append(new_array)
        # break
    # break
batchdata = np.concatenate(all_seg_data, axis=0)

In [62]:
np.concatenate(all_seg_data, axis=0).shape

(675, 1500, 5)

# test on autoencoder

In [66]:

import matplotlib.pyplot as plt

import torch.optim as optim

import numpy as np
import torch


from deepview.calculate_results.models.utils import (
    Resnet,
load_weights,
MSEloss,
# torch,
AE_eval_time_series,
AE_train_time_series_resnet,
AE_train_time_series,
majority_value,
# np,
adjust_learning_rate,
# tqdm
data_loader_umineko,
Autoencoder3d,
Autoencoder1d,
Autoencoder2d
)
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

In [65]:
data_b = batchdata[:,:,:len(select_columns)-1]
label_b = batchdata[:,:,len(select_columns)-1]
device = 'cuda'
batch_size = 128
train_set_r = data_loader_umineko(data_b, label_b, device=device)
train_loader = DataLoader(train_set_r, batch_size=batch_size,
                              shuffle=False, drop_last=False)

model = Autoencoder3d()
model = model.to(device)

criterion = MSEloss()
criterion = criterion.to(device)

learning_rate = 0.0001
optimizer = optim.Adam(
    model.parameters(), lr=learning_rate, amsgrad=True
)
optimizer = optim.Adam(
            model.parameters(), lr=learning_rate, amsgrad=True
        )
lambda1 = lambda epoch: 1.0**epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda1)



In [67]:
 # training
start_epoch = 0
num_epochs = 1000
# lr = 0.0001
training_loss = []
for epoch in tqdm(range(start_epoch, num_epochs)):

    learning_rate = adjust_learning_rate(
        learning_rate, optimizer, epoch, p_scheduler='cosine', p_epochs=num_epochs)

    losses = AE_train_time_series_resnet(train_loader, model, criterion, optimizer, epoch, scheduler, device)
    training_loss.append(losses.val)
    # if (epoch % 10 == 0) or (epoch == num_epochs - 1):
    #     print('loss of the ' + str(epoch) + '-th training epoch is :' + losses.__str__())

# 
# print('Saving model at: ' + 'AE_reconstruct_epoch%s' % str(epoch) \
#       + '_datalen%s_' % str(len_sw) +sensor_type+ '.pth')
# torch.save(model.state_dict(), 'AE_reconstruct_epoch%s' % str(epoch) + \
#            '_datalen%s_' % str(len_sw) +sensor_type+ '.pth')

plt.plot(training_loss)
plt.title('AE reconstruction loss of umineko2018 back1')
plt.show()

  0%|          | 0/1000 [00:17<?, ?it/s]


RuntimeError: Given groups=1, weight of size [64, 3, 3], expected input[128, 1500, 3] to have 3 channels, but got 1500 channels instead

In [ ]:
# reconstruction result
representation_list, sample_list, pred_list, label_list = \
            AE_eval_time_series(train_loader, model, device)

# tsne latent representation to shape=(2, len) PCA降维到形状为 (2, len)
repre_concat = np.concatenate(representation_list)
repre_reshape = repre_concat.reshape(repre_concat.shape[0], -1)

sample_concat = np.concatenate(sample_list)
# sample_reshape = sample_concat.reshape(-1, 3)
sample_concat = sample_concat.transpose(0,2,1)
sample_reshape = sample_concat.reshape(-1, sample_concat.shape[-1])

pred_concat = np.concatenate(pred_list)
# pred_reshape = pred_concat.reshape(-1, 3)
pred_concat = pred_concat.transpose(0, 2, 1)
pred_reshape = pred_concat.reshape(-1, pred_concat.shape[-1])

label_concat = np.concatenate(label_list)
label_concat_vote = majority_value(label_concat)
# label_concat_vote.shape

start, end = 1000, 2000
fig, axes = plt.subplots(3, 1, figsize=(8, 6)) 
axes[0].plot(sample_reshape[start:end, 1], 'r', label='groundtruthY')
axes[0].plot(pred_reshape[start:end, 1], 'b-.', label='predictY')
axes[0].set_title('Autoencoder_Reconstruct_Umineko2018_back1_accel_axisy')
axes[0].set_xlabel('timestamp')
axes[0].set_ylabel('ACC[G]')
axes[0].legend()

axes[1].plot(sample_reshape[start:end, 1], 'r', label='groundtruthY')
# axes[1].plot(pred_reshape[10000:length, 1], 'b-.', label='predictY')
# axes[1].set_title('ResNet_SSL_pretrained_Reconstruct_Umineko2018_back1')
axes[1].set_xlabel('timestamp')
axes[1].set_ylabel('ACC[G]')
axes[1].legend()

# axes[2].plot(sample_reshape[10000:length, 1], 'r', label='groundtruthY')
axes[2].plot(pred_reshape[start:end, 1], 'b-.', label='predictY')
# axes[2].set_title('ResNet_SSL_pretrained_Reconstruct_Umineko2018_back1')
axes[2].set_xlabel('timestamp')
axes[2].set_ylabel('ACC[G]')
axes[2].legend()

# Adjust layout
plt.tight_layout()
# Show the figure
plt.show()